# 01 — Dataset loading and inspection

**Source:** `Dataset/Phishing_URL_Dataset.csv` (PhiUSIIL), 235,795 rows, 55 columns.
SHA-256 `748ae7c7a677ac4ffc0eeefbee653dcef1d685b134ad2f5376596b92d79438ba`

## Label convention

`0 = Legitimate`, `1 = Phishing`. This is the mapping the whole project uses.
Any notebook or report claiming the reverse is wrong.

## 1.1 Environment

In [ ]:
import sys
from pathlib import Path

BACKEND = Path(r"E:\UIU\TRIMESTER - 11\Computer Security\CS_Project\Backend")
DATASET = BACKEND.parent / "Dataset"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

SEED = 42
np.random.seed(SEED)

print("backend :", BACKEND)
print("dataset :", DATASET)
print("numpy   :", np.__version__)
print("pandas  :", pd.__version__)

In [ ]:
def show(title, obj):
    print(f"\n=== {title} ===")
    print(obj)

def hr(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

## 1.2 Verify the dataset file

Hash first. If the digest does not match, every downstream number is suspect and
you should stop rather than train on an unknown file.

In [ ]:
import hashlib

DATASET_CSV = DATASET / "Phishing_URL_Dataset.csv"
EXPECTED_SHA256 = "748ae7c7a677ac4ffc0eeefbee653dcef1d685b134ad2f5376596b92d79438ba"

show("dataset file", {
    "path": str(DATASET_CSV),
    "exists": DATASET_CSV.exists(),
    "size_bytes": DATASET_CSV.stat().st_size if DATASET_CSV.exists() else None,
})

h = hashlib.sha256()
with DATASET_CSV.open("rb") as fh:
    for chunk in iter(lambda: fh.read(1 << 20), b""):
        h.update(chunk)
digest = h.hexdigest()

show("sha256", {"actual": digest, "expected": EXPECTED_SHA256, "match": digest == EXPECTED_SHA256})
assert digest == EXPECTED_SHA256, "Dataset hash mismatch — stop and investigate."

## 1.3 Column detection

`training/inspect_dataset.py` picks the URL and label columns by *content*, not by
name: the URL column must parse as an absolute URL at a rate of >= 0.5. This is why
the schemeless second dataset in `Dataset/` is rejected outright.

In [ ]:
from training.inspect_dataset import detect_columns

url_col, label_col, evidence = detect_columns(DATASET_CSV)
show("detected columns", {"url_column": url_col, "label_column": label_col})
show("evidence", evidence)

## 1.4 Load and inspect

In [ ]:
df = pd.read_csv(DATASET_CSV, usecols=[url_col, label_col], dtype=str)

show("shape", df.shape)
show("label value counts", df[label_col].value_counts(dropna=False))
show("nulls", df.isna().sum().to_dict())
show("duplicate URLs", int(df[url_col].duplicated().sum()))
show("head", df.head())

## 1.5 Label semantics — check this, do not assume

Map raw labels to integers with the project's own function rather than a hand-rolled
`astype(int)`. It raises on anything it cannot interpret instead of silently guessing.

In [ ]:
from app.preprocessing.url_dataset import to_label_array

y = to_label_array(df[label_col].tolist())
hr("label mapping")
print("1 = phishing, 0 = legitimate")
show("counts", pd.Series(y).value_counts().sort_index().to_dict())
show("phishing fraction", round(float(y.mean()), 6))

df = df.assign(label=y)
show("head with mapped labels", df.head())

## 1.6 Host and registered-domain structure

In [ ]:
from app.preprocessing.url_features import registered_domain_of
from app.preprocessing.url_preprocessing import host_of

hosts = df[url_col].map(host_of)
regs = df[url_col].map(registered_domain_of)

show("unparseable hosts", int((hosts == "").sum()))
show("unique hosts", int(hosts.nunique()))
show("unique registered domains", int(regs.nunique()))

# A host carrying both labels is a label-noise signal in PhiUSIIL.
lab = pd.DataFrame({"host": hosts, "label": df["label"]})
conflict = lab.groupby("host")["label"].nunique()
show("hosts with BOTH labels", int((conflict > 1).sum()))

## 1.7 Class balance by registered domain

In [ ]:
per_reg = pd.DataFrame({"registered_domain": regs, "label": df["label"]}) \
    .groupby("registered_domain")["label"].agg(["size", "mean"])

hr("registered-domain level")
show("domains", int(len(per_reg)))
show("single-row domains", int((per_reg["size"] == 1).sum()))
show("pure-phishing domains", int((per_reg["mean"] == 1.0).sum()))
show("pure-legitimate domains", int((per_reg["mean"] == 0.0).sum()))

# The reason the split groups by domain: random row-level splitting lets the
# same brand appear in train and test, which inflates scores.
show("top domains by row count", per_reg.sort_values("size", ascending=False).head(10))

> **Note.** These notebooks *call* the production modules in `app/` and
> `training/`. They do not reimplement the pipeline. The logic under test lives
> in exactly one place, so a notebook can never drift from what the CLI scripts
> and the API actually run.

## Next

`02_url_preprocessing.ipynb` — tokenizer, 30 handcrafted features, scaler.